# Lab 3 — Streaming & Incremental Ingestion with Auto Loader
**Goal:** work with streaming and incremental ingestion, and introduce schema evolution, using Databricks Auto Loader and Structured Streaming.

This notebook covers **Stage 1: Auto Loader**. Stage 2 (Event Hub streaming ingestion) follows in the same workspace / a follow-up notebook.

## 1. Environment Setup
Reset the source/schema/checkpoint volumes and drop the target table, so every run of this notebook starts from a clean slate.

In [0]:
source_path = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/source"
schema_path = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/schema"
checkpoint_path = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/checkpoints"


dbutils.fs.rm(source_path, True)
dbutils.fs.rm(schema_path, True)
dbutils.fs.rm(checkpoint_path, True)

dbutils.fs.mkdirs(source_path)
dbutils.fs.mkdirs(schema_path)
dbutils.fs.mkdirs(checkpoint_path)


spark.sql("""
DROP TABLE IF EXISTS 
dbr_dev.ayyuborujzade_bronze.autoloader_events
""")

## 2. Upload a Large Number of Files
Generate 10,000 synthetic click events and write them out as **1,000 separate JSON files** (`repartition(1000)`) to simulate a real-world "many small files" ingestion scenario.


In [0]:
from pyspark.sql import Row
import random

data = []
for i in range(10000):

    data.append(
        Row(
            id=i,
            event="click",
            value=random.randint(1,1000)
        )
    )


df = spark.createDataFrame(data)

df.repartition(1000) \
.write \
.mode("append") \
.json(source_path)

## 3. Ingest Data with Auto Loader & Structured Streaming
Define the Auto Loader read stream against the source volume, with:
- `cloudFiles.schemaLocation` — where Auto Loader persists its inferred/evolving schema
- `cloudFiles.schemaEvolutionMode = "addNewColumns"` — new columns get added automatically rather than failing the pipeline outright

The write side uses `trigger(availableNow=True)` — process everything currently available, then stop. This is the cost-efficient pattern for scheduled/batch-style incremental loads (cluster doesn't need to stay up between runs).

In [0]:
stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        schema_path
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "addNewColumns"
    )
    .load(source_path)
)

In [0]:
query = (
    stream_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        "dbr_dev.ayyuborujzade_bronze.autoloader_events"
    )
)

**Verify the first batch landed correctly:**

In [0]:
spark.table(
"dbr_dev.ayyuborujzade_bronze.autoloader_events"
).printSchema()

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM dbr_dev.ayyuborujzade_bronze.autoloader_events
""").show()

## 4. Schema Inference & Evolution — Adding a New Column
Append two new records that include a column (`country`) not present in the original data, then re-ingest.

In [0]:
new_data = [
    (10001,"purchase",500,"Poland"),
    (10002,"payment",300,"Germany")
]

new_df = spark.createDataFrame(
    new_data,
    [
        "id",
        "event",
        "value",
        "country"
    ]
)


new_df.write \
.mode("append") \
.json(source_path)

### Why this needs a retry loop
With `schemaEvolutionMode = "addNewColumns"`, the first time Auto Loader's stream sees a field that isn't in its stored schema (`country`), it's designed to throw `UnknownFieldException` and stop the query — this is **intentional**, not a bug: Auto Loader is halting so it can update its schema log. The error message says so directly: *"which can be fixed by an automatic retry: true"*.

In a scheduled **Databricks Job**, this exception is caught and the stream auto-restarts for you. In an interactive notebook there's no built-in auto-restart, so the cell below wraps the read+write in a small retry loop that catches specifically this schema-evolution exception, logs that it happened, and retries — mirroring what a Job does automatically, instead of requiring a manual rerun.

In [0]:
def start_autoloader_batch():
    stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path)
    )
    return (
        stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("dbr_dev.ayyuborujzade_bronze.autoloader_events")
    )

max_retries = 2
query = None

for attempt in range(1, max_retries + 1):
    try:
        query = start_autoloader_batch()
        query.awaitTermination()
        print(f"Batch completed successfully on attempt {attempt}.")
        break
    except Exception as e:
        # This specific exception means Auto Loader detected a new column, updated its
        # schema log, and needs a fresh stream/query to pick that schema up - expected
        # behavior for addNewColumns, not a real failure.
        if "UnknownFieldException" in str(e) or "NEW_FIELDS_IN_RECORD" in str(e):
            print(
                f"Attempt {attempt}: new field detected mid-stream (schema evolution). "
                "Auto Loader updated its schema log - retrying with a fresh stream."
            )
            continue
        raise

**Verify the schema evolved and the new rows landed:**

In [0]:
spark.table(
"dbr_dev.ayyuborujzade_bronze.autoloader_events"
).printSchema()

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM dbr_dev.ayyuborujzade_bronze.autoloader_events
""").show()

In [0]:
display(
spark.sql("""
SELECT *
FROM dbr_dev.ayyuborujzade_bronze.autoloader_events
WHERE country IS NOT NULL
""")
)

## 5. Streaming Progress & Checkpoint Inspection
`query.lastProgress` gives the metrics for the most recent micro-batch; the checkpoint directory listing confirms Structured Streaming is persisting the offsets/commits it needs for safe restarts.

In [0]:
query.lastProgress

In [0]:
display(
dbutils.fs.ls(checkpoint_path)
)

Stage 2

## 6. Analyzing Streaming Statistics (Files/Rows per Batch)
Each streaming micro-batch commits as one Delta table version, so `DESCRIBE HISTORY` gives a reliable per-batch view of rows and files written by Auto Loader. Note: the metric key for file count under `STREAMING UPDATE` operations is `numAddedFiles` (not `numFiles`, which is only populated for plain `WRITE` operations).

In [0]:
history_df = spark.sql("""
DESCRIBE HISTORY dbr_dev.ayyuborujzade_bronze.autoloader_events
""")

display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics.numOutputRows",
        "operationMetrics.numAddedFiles"
    ).orderBy("version")
)

An alternative view is the streaming query's own progress log (`query.recentProgress`). **Caveat:** this only reflects whatever query object is currently held in the `query` variable — if Auto Loader restarted internally partway through (as it does on schema evolution), this view can look sparse/incomplete for that run. `DESCRIBE HISTORY` above is the more reliable source of truth for historical batch statistics.

In [0]:
for batch in query.recentProgress:
    src = batch["sources"][0] if batch.get("sources") else {}
    print(
        f"batchId={batch['batchId']:>3}  "
        f"numInputRows={batch['numInputRows'] if batch['numInputRows'] is not None else 0:>6}  "
        f"triggerExecutionMs={batch['durationMs'].get('triggerExecution')}  "
        f"filesOutstanding={src.get('metrics', {}).get('numFilesOutstanding')}"
    )

## 7. Observing the `_rescued_data` Column
To confirm Auto Loader's safety net actually works (not just an unused column), deliberately write a record with a field that doesn't fit the existing schema shape and re-ingest.

In [0]:
malformed_json = '{"id": 99999, "event": "click", "value": {"nested": "oops"}, "country": "TestCountry"}\n'

dbutils.fs.put(source_path + "/malformed_manual.json", malformed_json, overwrite=True)

In [0]:
stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

query = (
    stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("dbr_dev.ayyuborujzade_bronze.autoloader_events")
)
query.awaitTermination()

In [0]:
display(
spark.sql("""
SELECT id, event, value, country, _rescued_data
FROM dbr_dev.ayyuborujzade_bronze.autoloader_events
WHERE _rescued_data IS NOT NULL
""")
)

**Result: this comes back empty, and that's expected.** Looking at the schema output above, `event`, `id`, and `value` were all inferred as **string**. Spark's JSON reader treats `string` as a catch-all: it will serialize *any* JSON value (including a nested object) into a string field rather than flagging a mismatch - there's nothing to rescue from a column type that accepts everything. Section 11 below revisits this with proper type inference enabled, where a genuine mismatch — and a populated `_rescued_data` — actually occurs.

## 8. Experimenting with Trigger Types
- **`availableNow=True`** (used above): processes everything currently available, then stops. Cheapest option since compute doesn't need to stay up between runs — best fit for scheduled incremental batch jobs.
- **`processingTime="N seconds"`**: keeps the query alive, checking for new files on a fixed interval. Needed for near-real-time ingestion, at the cost of the cluster having to stay up.

A short-lived `processingTime` stream is run below against a disposable table just to demonstrate the syntax/behavior difference, then stopped after ~20 seconds so it doesn't run indefinitely (cost awareness).

In [0]:
demo_source = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/demo_source"
demo_schema = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/demo_schema"
demo_checkpoint = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/demo_checkpoint"

dbutils.fs.rm(demo_source, True)
dbutils.fs.rm(demo_schema, True)
dbutils.fs.rm(demo_checkpoint, True)
dbutils.fs.mkdirs(demo_source)

spark.sql("DROP TABLE IF EXISTS dbr_dev.ayyuborujzade_bronze.autoloader_trigger_demo")

from pyspark.sql import Row
seed_data = [Row(id=i, event="test", value=i * 10) for i in range(5)]
spark.createDataFrame(seed_data).repartition(5).write.mode("append").json(demo_source)

In [0]:
processing_time_query = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", demo_schema)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(demo_source)
    .writeStream
    .format("delta")
    .option("checkpointLocation", demo_checkpoint)
    .trigger(processingTime="10 seconds")
    .toTable("dbr_dev.ayyuborujzade_bronze.autoloader_trigger_demo")
)

processing_time_query.awaitTermination(20)
processing_time_query.stop()

spark.sql(
    "SELECT COUNT(*) FROM dbr_dev.ayyuborujzade_bronze.autoloader_trigger_demo"
).show()

In [0]:
def run_checkpoint_demo():
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", demo_schema)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(demo_source)
    )
    q = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", demo_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("dbr_dev.ayyuborujzade_bronze.autoloader_checkpoint_demo")
    )
    q.awaitTermination()
    return q

spark.sql("DROP TABLE IF EXISTS dbr_dev.ayyuborujzade_bronze.autoloader_checkpoint_demo")
dbutils.fs.rm(demo_checkpoint, True)

run_checkpoint_demo()
print("After first run:")
spark.sql("SELECT COUNT(*) FROM dbr_dev.ayyuborujzade_bronze.autoloader_checkpoint_demo").show()

## 9. Checkpoint-Based Safe Reload Test
A genuinely "safe" reload means rerunning the same stream with no new files should **not** duplicate rows, because the checkpoint remembers which files were already processed. To show why the checkpoint matters, the checkpoint is then deleted and the stream rerun — which *does* reprocess everything and creates duplicates.

In [0]:
run_checkpoint_demo()
print("After re-running with the SAME checkpoint (no new files):")
spark.sql("SELECT COUNT(*) FROM dbr_dev.ayyuborujzade_bronze.autoloader_checkpoint_demo").show()

In [0]:
dbutils.fs.rm(demo_checkpoint, True)

run_checkpoint_demo()
print("After DELETING the checkpoint and re-running (files reprocessed as if new):")
spark.sql("SELECT COUNT(*) FROM dbr_dev.ayyuborujzade_bronze.autoloader_checkpoint_demo").show()

In [0]:

spark.sql("DROP TABLE IF EXISTS dbr_dev.ayyuborujzade_bronze.autoloader_checkpoint_demo")
spark.sql("DROP TABLE IF EXISTS dbr_dev.ayyuborujzade_bronze.autoloader_trigger_demo")
dbutils.fs.rm(demo_source, True)
dbutils.fs.rm(demo_schema, True)
dbutils.fs.rm(demo_checkpoint, True)

## 10. Rescued-Data Demo, Revisited (Properly Typed)
Section 7 came back empty because every column in the main pipeline is a string, and a string column accepts anything. To see a genuine rescue, at least one column needs a real, non-string type so a bad value has something concrete to conflict with — that requires `cloudFiles.inferColumnTypes = "true"`. The main pipeline's schema is already locked in as all-string from its first batch and won't retype now, so this is a small, self-contained demo on its own paths/table with type inference turned on from the start.

In [0]:
rescue_demo_source = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/rescue_demo_source"
rescue_demo_schema = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/rescue_demo_schema"
rescue_demo_checkpoint = "/Volumes/dbr_dev/ayyuborujzade_bronze/lab3_volume/rescue_demo_checkpoint"

dbutils.fs.rm(rescue_demo_source, True)
dbutils.fs.rm(rescue_demo_schema, True)
dbutils.fs.rm(rescue_demo_checkpoint, True)
dbutils.fs.mkdirs(rescue_demo_source)

spark.sql("DROP TABLE IF EXISTS dbr_dev.ayyuborujzade_bronze.autoloader_rescue_demo")

good_json = (
    '{"id": 1, "event": "click", "value": 100}\n'
    '{"id": 2, "event": "click", "value": 200}\n'
)
dbutils.fs.put(rescue_demo_source + "/seed.json", good_json, overwrite=True)

In [0]:
rescue_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", rescue_demo_schema)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(rescue_demo_source)
)

rescue_query = (
    rescue_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", rescue_demo_checkpoint)
    .trigger(availableNow=True)
    .toTable("dbr_dev.ayyuborujzade_bronze.autoloader_rescue_demo")
)
rescue_query.awaitTermination()

spark.table("dbr_dev.ayyuborujzade_bronze.autoloader_rescue_demo").printSchema()

In [0]:
bad_json = '{"id": 3, "event": "click", "value": "not_a_number"}\n'
dbutils.fs.put(rescue_demo_source + "/bad_record.json", bad_json, overwrite=True)

rescue_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", rescue_demo_schema)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(rescue_demo_source)
)

rescue_query = (
    rescue_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", rescue_demo_checkpoint)
    .trigger(availableNow=True)
    .toTable("dbr_dev.ayyuborujzade_bronze.autoloader_rescue_demo")
)
rescue_query.awaitTermination()

In [0]:
display(
spark.sql("""
SELECT id, event, value, _rescued_data
FROM dbr_dev.ayyuborujzade_bronze.autoloader_rescue_demo
WHERE _rescued_data IS NOT NULL
""")
)

In [0]:
spark.sql("DROP TABLE IF EXISTS dbr_dev.ayyuborujzade_bronze.autoloader_rescue_demo")
dbutils.fs.rm(rescue_demo_source, True)
dbutils.fs.rm(rescue_demo_schema, True)
dbutils.fs.rm(rescue_demo_checkpoint, True)

## Stage 1 Summary — Done-When Checklist
- **Auto Loader ingests incrementally**: `trigger(availableNow=True)` picks up only newly arrived files on each run (Sections 3–4), confirmed via row counts and `DESCRIBE HISTORY`.
- **A newly added source column is handled via schema evolution without failing**: adding `country` triggers `UnknownFieldException` (by design), which is caught and retried automatically in Section 4 — the pipeline ends up with the new column merged in, no manual intervention required.
- **The rescued-data column works as intended**: Section 7 shows the realistic case where string-typed columns don't need rescuing; Section 10 proves the mechanism itself works correctly once a real type conflict exists.
- **Streaming statistics are analyzed** per batch via `DESCRIBE HISTORY` (Section 6).
- **Multiple trigger types are compared**: `availableNow` vs `processingTime` (Section 8).
- **A checkpoint-based reload is proven safe**: rerunning with the same checkpoint adds no duplicates; deleting the checkpoint does (Section 9) — demonstrating why the checkpoint path must be treated as durable state.